# Theta/alpha power clustering

## Scientific question

Can baseline-corrected theta and alpha power reveal reproducible condition differences while also defining one connected space-time region per band that is large enough to reduce the input dimensionality of SVC and EEGNet?

The inferential analysis and the engineering reduction use the same power tensors but answer different questions. FWER-corrected clusters test evidence against the null; the connected reduction region retains a fixed, interpretable fraction of high-effect points even when no cluster is significant.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from br_analysis.cluster_analysis import (
    jaccard_divergence,
    load_cluster_result,
    run_power_cluster_analysis,
    save_cluster_result,
    subject_condition_means,
    subject_reduction_regions,
)
from br_analysis.config import ProjectPaths
from br_analysis.power import PowerConfig, load_power_dataset
from br_analysis.reduction import channel_coordinates
from br_analysis.tasks import get_task

## Configuration

Power is estimated independently for theta (5-7 Hz) and alpha (8-12 Hz) with a 500 ms Hanning window. Every trial is converted to dB relative to its own -740 to -240 ms baseline. The displayed samples are approximately 20 ms apart.

`FORCE_CLUSTERING=False` reuses the single saved result for each task and band. Set it to `True` only when intentionally replacing those results. Use 10,000 permutations for the final report; 2,000 is a faster exploratory default.

In [ ]:
TASK_ID = "c_vs_ip"
N_PERMUTATIONS = 2_000
P_UNCORRECTED = 0.05
REDUCTION_FRACTION = 0.05
MIN_REDUCTION_CHANNELS = 3
MIN_REDUCTION_DURATION = 0.25
FORCE_CLUSTERING = False

paths = ProjectPaths.from_env("clustering", create_dirs=True)
task = get_task(TASK_ID)
power_config = PowerConfig()
result_dir = paths.artifacts_dir / "clustering" / task.id

## Selected task, classes and cohort

The power cache is shared across tasks. Changing `TASK_ID` only changes event-to-class mapping and subject inclusion; it does not recompute power when compatible subject caches already exist.

In [ ]:
dataset = load_power_dataset(paths, task, config=power_config)
cohort_rows = []
for subject in dataset.subject_ids:
    subject_mask = dataset.subjects == subject
    row = {"subject": subject, "trials": int(subject_mask.sum())}
    row.update({class_id: int(np.sum(subject_mask & (dataset.y == index))) for index, class_id in enumerate(dataset.class_ids)})
    cohort_rows.append(row)

display(pd.DataFrame(cohort_rows))
display(pd.DataFrame([{
    "task": task.id,
    "classes": " vs ".join(task.class_ids),
    "subjects": len(dataset.subject_ids),
    "trials": len(dataset.y),
    "bands": ", ".join(dataset.band_names),
    "channels": len(dataset.channel_names),
    "power_time_points": len(dataset.times),
}]))

## Separate band analyses

For each band, trials are first averaged within subject and condition. The contrast therefore has one observation per subject, avoiding trial-level pseudoreplication.

Two complementary analyses follow:

- **Article ROI test:** theta uses Fz/FCz/Cz and alpha uses posterior-occipital electrodes. A temporal cluster permutation test reports FWER-corrected p-values.
- **Whole-head space-time test:** adjacency joins neighbouring electrodes and consecutive samples. Its FWER values are reported as evidence, but significance is not required for reduction.
- **Reduction region:** one connected region grows from the strongest smoothed absolute t-value until it retains at least 5% of points, three channels and 250 ms. Theta and alpha are never clustered together.

In [ ]:
results = {}
for band in dataset.band_names:
    result_path = result_dir / f"{band}.npz"
    if result_path.is_file() and not FORCE_CLUSTERING:
        print(f"{band}: loading {result_path}")
        results[band] = load_cluster_result(result_path)
    else:
        print(f"{band}: running {N_PERMUTATIONS:,} permutations")
        result = run_power_cluster_analysis(
            dataset,
            band,
            n_permutations=N_PERMUTATIONS,
            p_uncorrected=P_UNCORRECTED,
            reduction_fraction=REDUCTION_FRACTION,
            min_reduction_channels=MIN_REDUCTION_CHANNELS,
            min_reduction_duration=MIN_REDUCTION_DURATION,
        )
        save_cluster_result(result, result_path)
        results[band] = result

## FWER-corrected cluster summary

Small significant clusters remain valid inferential findings, but they are not used directly as the model representation. The reduction row describes the larger connected region intended for dimensionality reduction.

In [ ]:
summary_rows = []
for band, result in results.items():
    for scope, clusters, p_values in (
        ("article_roi", result.roi_clusters, result.roi_cluster_p_values),
        ("whole_head", result.clusters, result.cluster_p_values),
    ):
        for cluster_index, (mask, p_value) in enumerate(zip(clusters, p_values), start=1):
            time_mask = mask if mask.ndim == 1 else mask.any(axis=0)
            occupied = np.flatnonzero(time_mask)
            summary_rows.append({
                "band": band,
                "scope": scope,
                "cluster": cluster_index,
                "p_fwer": float(p_value),
                "significant": bool(p_value <= 0.05),
                "points": int(mask.sum()),
                "tmin_ms": float(dataset.times[occupied[0]] * 1_000) if len(occupied) else np.nan,
                "tmax_ms": float(dataset.times[occupied[-1]] * 1_000) if len(occupied) else np.nan,
            })
    region = result.reduction_region
    occupied = np.flatnonzero(region.any(axis=0))
    summary_rows.append({
        "band": band,
        "scope": "reduction_region",
        "cluster": 1,
        "p_fwer": np.nan,
        "significant": np.nan,
        "points": int(region.sum()),
        "tmin_ms": float(dataset.times[occupied[0]] * 1_000),
        "tmax_ms": float(dataset.times[occupied[-1]] * 1_000),
    })

cluster_summary = pd.DataFrame(summary_rows)
display(cluster_summary.sort_values(["band", "scope", "p_fwer"], na_position="last"))

## Article ROI power time courses

Curves show subject-level condition means and 95% normal-approximation confidence intervals. Dark bars mark ROI temporal clusters with `p_FWER <= 0.05`. The sign is always the first configured class minus the second.

In [ ]:
fig, axes = plt.subplots(len(dataset.band_names), 1, figsize=(11, 7), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)
colors = sns.color_palette("colorblind", len(dataset.class_ids))
for axis, band in zip(axes, dataset.band_names):
    result = results[band]
    class_means, _ = subject_condition_means(dataset, band)
    roi_indices = [dataset.channel_names.index(channel) for channel in result.roi_channels]
    roi_means = class_means[:, :, roi_indices].mean(axis=2)
    for class_index, (class_id, color) in enumerate(zip(dataset.class_ids, colors)):
        values = roi_means[:, class_index]
        mean = values.mean(axis=0)
        error = 1.96 * values.std(axis=0, ddof=1) / np.sqrt(len(values))
        axis.plot(dataset.times, mean, label=class_id, color=color)
        axis.fill_between(dataset.times, mean - error, mean + error, color=color, alpha=0.2)
    ymin, ymax = axis.get_ylim()
    bar_y = ymin + 0.04 * (ymax - ymin)
    for mask, p_value in zip(result.roi_clusters, result.roi_cluster_p_values):
        if p_value <= 0.05:
            axis.plot(dataset.times[mask], np.full(mask.sum(), bar_y), color="black", linewidth=5)
    axis.axhline(0, color="0.5", linewidth=0.8)
    axis.set(title=f"{band.title()} ROI: {', '.join(result.roi_channels)}", ylabel="Power change (dB)")
axes[-1].set_xlabel("Time from stimulus (s)")
axes[0].legend(frameon=False)
plt.show()

## Whole-head statistic and connected reduction region

The heatmaps show paired t-values across all channels and times. The black contour is the single connected region retained for model reduction; it is intentionally independent of whether an inferential cluster crosses the FWER threshold.

In [ ]:
fig, axes = plt.subplots(len(dataset.band_names), 1, figsize=(13, 9), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)
limit = max(float(np.nanmax(np.abs(result.statistic))) for result in results.values())
for axis, band in zip(axes, dataset.band_names):
    result = results[band]
    image = axis.imshow(
        result.statistic,
        aspect="auto",
        origin="lower",
        extent=[dataset.times[0], dataset.times[-1], 0, len(dataset.channel_names) - 1],
        cmap="RdBu_r",
        vmin=-limit,
        vmax=limit,
    )
    axis.contour(dataset.times, np.arange(len(dataset.channel_names)), result.reduction_region, levels=[0.5], colors="black", linewidths=1.2)
    tick_indices = np.linspace(0, len(dataset.channel_names) - 1, 10, dtype=int)
    axis.set_yticks(tick_indices, np.asarray(dataset.channel_names)[tick_indices])
    axis.set(title=f"{band.title()}: class 1 - class 2 paired t-map", ylabel="Channel")
axes[-1].set_xlabel("Time from stimulus (s)")
fig.colorbar(image, ax=axes, label="paired t")
plt.show()

## Spatial footprint and dimensionality

Occupancy is the percentage of post-stimulus samples retained for each channel. The two band masks can be concatenated for SVC features or used as fixed masks for an EEGNet input tensor.

In [ ]:
coordinates = channel_coordinates(dataset.channel_names)
fig, axes = plt.subplots(1, len(dataset.band_names), figsize=(11, 4), constrained_layout=True)
axes = np.atleast_1d(axes)
for axis, band in zip(axes, dataset.band_names):
    occupancy = results[band].reduction_region.mean(axis=1) * 100
    points = axis.scatter(coordinates[:, 0], coordinates[:, 1], c=occupancy, s=80, cmap="viridis", vmin=0)
    for channel, (x, y) in zip(dataset.channel_names, coordinates[:, :2]):
        if occupancy[dataset.channel_names.index(channel)] > 0:
            axis.text(x, y, channel, fontsize=7, ha="center", va="center")
    axis.set(title=f"{band.title()} retained time", aspect="equal")
    axis.axis("off")
fig.colorbar(points, ax=axes, label="Retained samples (%)")
plt.show()

full_points = len(dataset.band_names) * len(dataset.channel_names) * len(dataset.times)
retained_points = sum(result.reduction_region.sum() for result in results.values())
display(pd.DataFrame([{
    "full_band_power_points": full_points,
    "retained_connected_points": int(retained_points),
    "retained_percent": 100 * retained_points / full_points,
    "compression_ratio": full_points / retained_points,
}]))

## Subject stability and Jaccard divergence

Each subject receives a region with the same connectivity and size rules, using only that subject's condition contrast. Jaccard divergence against the group region quantifies heterogeneity; high divergence is descriptive and does not invalidate the group inferential test.

In [ ]:
jaccard_rows = []
for band, result in results.items():
    individual_regions = subject_reduction_regions(
        result,
        dataset.channel_names,
        dataset.times,
        target_fraction=REDUCTION_FRACTION,
        min_channels=MIN_REDUCTION_CHANNELS,
        min_duration=MIN_REDUCTION_DURATION,
    )
    for subject, region in zip(dataset.subject_ids, individual_regions):
        jaccard_rows.append({
            "band": band,
            "subject": subject,
            "jaccard_divergence": jaccard_divergence(region, result.reduction_region),
        })

jaccard_table = pd.DataFrame(jaccard_rows)
display(jaccard_table.groupby("band")["jaccard_divergence"].describe())
sns.boxplot(data=jaccard_table, x="band", y="jaccard_divergence", hue="band", palette="colorblind", legend=False)
plt.ylim(0, 1)
plt.title("Individual versus group connected regions")
plt.show()

## Reproducibility and saved outputs

Prepare all expanded epochs and power tensors once:

```text
br-analysis prepare-data
```

The notebook stores one result per task and band under `artifacts/clustering/<task>/theta.npz` and `alpha.npz`. Reopening it loads those files unless `FORCE_CLUSTERING=True`. The power caches under `artifacts/_cache/power/` are shared by every task.

## Interpretation and limitations

FWER-corrected ROI clusters address replication of the article's condition effects. Whole-head clusters provide exploratory localisation. The connected reduction regions instead answer an engineering question and must be evaluated by held-out model performance against agnostic SVC/EEGNet baselines.

The reduction mask is currently estimated from all subjects, so it must not yet be used as if it were leakage-free cross-validation. After visually accepting the regions, model integration should either define them on an independent task/cohort or recompute them inside each training fold. The final inferential report should use 10,000 permutations.